# PROJECT AKHIR PENGOLAHAN CITRA DIGITAL 2026 (FINAL)
## Forensik Citra Struk Belanja untuk Deteksi Manipulasi Nominal Menggunakan Segmentasi Karakter dan Inkonsistensi Noise

**Kelompok 4:**
- Imanuel Carlos Lumban Tobing (2310511143)
- Abdul Faris Aufar (2310511154)
- Muhammad Ammar Allezandro Djosan (2310511128)

**Dosen Pengampu:** Dr. Mayanda Mega Santoni, S.Komp, M.Kom.

---

### Prinsip & Batasan Sistem
- **100% Pengolahan Citra Digital (PCD) Klasik** menggunakan Python + OpenCV + NumPy + scikit-image + SciPy (**tanpa YOLO dan tanpa Deep Learning**).
- **Tanpa Dokumen Asli Pembanding (*Blind Forgery Detection*):** Keputusan diambil hanya dari 1 citra struk scan yang diunggah.
- **Runtime Google Colab:** Cukup menggunakan **CPU standar** (tidak memerlukan GPU).

### Peta Isi Notebook (Step-by-Step)
1. **Setup & Ekstraksi Dataset** (*Find it! Again*, ICDAR 2023) dari Google Drive.
2. **Audit Kualitas & Statistik Dataset** (verifikasi 27 metrik vs paper, distribusi warna RGB/Grayscale, teknik & entitas).
3. **Bedah Visual Step-by-Step 8 Tahap PCD (*Before & After* Tiap Tahap):**
   - Tahap 1–2: Input & Pra-pemrosesan (`Grayscale + CLAHE` + Histogram *Before vs After*)
   - Tahap 3: Ruang Warna (`Sisa Warna / Chromatic Artifact`)
   - Tahap 4: Filtering (`Median Filter` & `Residu Noise`)
   - Tahap 5: Segmentasi (`Multi-Otsu thresholds[0] vs thresholds[1] + Adaptive Threshold`)
   - Tahap 6: Morfologi (`Opening` + `Connected Components` per Karakter + `Dilasi Horizontal` per Blok Kata)
   - Tahap 7: Ekstraksi Fitur Forensik (*Before/After* Bukti `CPI` Copy-Paste vs Huruf Kembar Asli & Bukti `IMI` *ClearType*)
   - Tahap 8: Output Akhir (*Before* Input Polos vs *After* Kotak Merah Area Curiga)
4. **Tuning Threshold pada Split `val`** (192 struk valid, bebas kebocoran data).
5. **Evaluasi Resmi pada Split `test`** (218 struk, 35 palsu) vs Baseline ICDAR 2023 + IoU + Analisis Per Teknik/Entitas/False Positive.
6. **Galeri *Before & After* Hasil Deteksi (`TP`, `FP`, `FN`) & Export `config_final.json`** untuk Aplikasi Streamlit.

---
## Bagian 1 — Setup Library, Mount Google Drive, & Ekstrak Dataset

In [ ]:
import os
import ast
import json
import time
import zipfile
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy import stats
from skimage.filters import threshold_multiotsu

plt.rcParams['figure.dpi'] = 110
print(f"OpenCV    : {cv2.__version__}")
print(f"NumPy     : {np.__version__}")
print(f"Pandas    : {pd.__version__}")
print(f"CPU cores : {os.cpu_count()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DRIVE_DIR = Path('/content/drive/MyDrive/PCD_Final Project')
ZIP_PATH = PROJECT_DRIVE_DIR / 'Dataset' / 'findit2.zip'
EXTRACT_DIR = Path('/content/data/findit2')

assert ZIP_PATH.exists(), f"Dataset tidak ditemukan di {ZIP_PATH}"
if not EXTRACT_DIR.exists() or not any(EXTRACT_DIR.iterdir()):
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print("Mengekstrak findit2.zip ke disk lokal Colab (lebih cepat dari baca langsung di Drive)...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print("Ekstraksi selesai!")
else:
    print("Dataset sudah diekstrak sebelumnya di /content/data/findit2.")

DATA_ROOT = EXTRACT_DIR / 'findit2' if (EXTRACT_DIR / 'findit2' / 'train.txt').exists() else EXTRACT_DIR
print(f"DATA_ROOT aktif: {DATA_ROOT}")

---
## Bagian 2 — Audit & Eksplorasi Dataset (*Find it! Again*, ICDAR 2023)

Kita memuat seluruh label split (`train`, `val`, `test`), mem-parsing anotasi kotak *VGG Image Annotator* pada kolom `forgery annotations`, memeriksa anomali kualitas data (file hilang/duplikat), serta membandingkan statistik dataset aktual dengan angka di paper ICDAR 2023.

In [ ]:
split_rows, via_rows, region_rows, missing_rows = [], [], [], []

for split_name in ['train', 'val', 'test']:
    df_s = pd.read_csv(DATA_ROOT / f'{split_name}.txt')
    df_s.columns = [c.strip().lower() for c in df_s.columns]
    img_dir = DATA_ROOT / split_name

    for _, r in df_s.iterrows():
        fname = str(r['image']).strip()
        fpath = img_dir / fname
        exists = fpath.exists()
        if not exists:
            missing_rows.append({'split': split_name, 'filename': fname, 'forged': int(r['forged'])})

        split_rows.append({
            'split': split_name,
            'filename': fname,
            'filepath': str(fpath) if exists else '',
            'exists': exists,
            'digital_annotation': int(r.get('digital annotation', 0)),
            'handwritten_annotation': int(r.get('handwritten annotation', 0)),
            'forged': int(r['forged'])
        })

        raw_ann = r.get('forgery annotations', '0')
        if pd.notna(raw_ann) and str(raw_ann).strip() not in {'0', '', 'nan', 'None'}:
            ann = ast.literal_eval(str(raw_ann).strip())
            f_attr = ann.get('file_attributes', {}) or {}
            sw_raw = str(f_attr.get('Software used', '')).strip().lower()
            sw_norm = 'preview' if sw_raw in {'aperçu', 'apercu', 'preview'} else sw_raw
            regions = ann.get('regions', []) or []
            via_rows.append({'split': split_name, 'filename': fname, 'software_used': sw_norm, 'num_regions': len(regions)})

            for idx, reg in enumerate(regions):
                s_attr = reg.get('shape_attributes', {}) or {}
                r_attr = reg.get('region_attributes', {}) or {}
                mod = r_attr.get('Modified area', {})
                if isinstance(mod, dict):
                    mod_types = [k.upper() for k, v in mod.items() if v and str(k).lower() != 'none']
                elif isinstance(mod, str) and mod and mod.lower() != 'none':
                    mod_types = [mod.upper()]
                else:
                    mod_types = []
                orig_flag = str(r_attr.get('Original area', 'no')).strip().lower()
                region_rows.append({
                    'split': split_name,
                    'filename': fname,
                    'x': int(s_attr.get('x', 0)),
                    'y': int(s_attr.get('y', 0)),
                    'width': int(s_attr.get('width', 0)),
                    'height': int(s_attr.get('height', 0)),
                    'mod_types': mod_types,
                    'entity_type': str(r_attr.get('Entity type', 'Unknown')).strip(),
                    'is_original_area': orig_flag in {'yes', 'true', '1'}
                })

df_splits = pd.DataFrame(split_rows)
df_via = pd.DataFrame(via_rows)
df_regions = pd.DataFrame(region_rows)

df_train = df_splits[(df_splits['split'] == 'train') & (df_splits['exists'])].reset_index(drop=True)
df_val   = df_splits[(df_splits['split'] == 'val')   & (df_splits['exists'])].reset_index(drop=True)
df_test  = df_splits[(df_splits['split'] == 'test')  & (df_splits['exists'])].reset_index(drop=True)

print(f"Total baris split : {len(df_splits)} | File gambar valid: {df_splits['exists'].sum()}")
print(f"Anomali file hilang di folder split: {missing_rows}")
print(f"  -> Train valid : {len(df_train)} ({df_train['forged'].sum()} palsu)")
print(f"  -> Val valid   : {len(df_val)} ({df_val['forged'].sum()} palsu) [1 file hilang dikecualikan]")
print(f"  -> Test valid  : {len(df_test)} ({df_test['forged'].sum()} palsu)")

In [ ]:
# Audit mode warna seluruh citra valid + visualisasi karakteristik dataset
color_modes = []
for fp in df_splits.loc[df_splits['exists'], 'filepath']:
    img = cv2.imread(fp, cv2.IMREAD_UNCHANGED)
    if img is None:
        continue
    if img.ndim == 2:
        color_modes.append('Grayscale (1-ch)')
    else:
        bgr = img[:, :, :3].astype(np.float32)
        diff = np.abs(bgr - bgr.mean(axis=2, keepdims=True)).max()
        color_modes.append('True Color (RGB)' if diff > 0 else 'Grayscale (3-ch)')

cm_s = pd.Series(color_modes).value_counts()
df_mod = df_regions[~df_regions['is_original_area']]
tech_s = pd.Series(Counter(t for lst in df_mod['mod_types'] for t in lst)).sort_values(ascending=True)
ent_s = df_mod['entity_type'].value_counts().sort_values(ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

axes[0].bar(cm_s.index, cm_s.values, color=['#64748b', '#2563eb', '#94a3b8'][:len(cm_s)])
axes[0].set_title('1. Mode Warna Citra (~60,6% Grayscale)')
axes[0].set_ylabel('Jumlah Struk')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(cm_s.values):
    axes[0].text(i, v + 8, f"{v} ({v/len(color_modes)*100:.1f}%)", ha='center', fontsize=9, fontweight='bold')

axes[1].barh(tech_s.index, tech_s.values, color=['#dc2626' if k == 'CPI' else '#f59e0b' if k == 'IMI' else '#64748b' for k in tech_s.index])
axes[1].set_title('2. Teknik Pemalsuan (Total 466 Area)')
axes[1].set_xlabel('Jumlah Area')
for i, v in enumerate(tech_s.values):
    axes[1].text(v + 5, i, str(v), va='center', fontweight='bold')

axes[2].barh(ent_s.index, ent_s.values, color=['#dc2626' if 'total' in str(k).lower() else '#3b82f6' for k in ent_s.index])
axes[2].set_title('3. Jenis Isi yang Diubah (Total/Payment Dominan)')
axes[2].set_xlabel('Jumlah Area')
for i, v in enumerate(ent_s.values):
    axes[2].text(v + 4, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

**Temuan kunci audit data:**
- **60,6% citra adalah Grayscale**, hanya **39,4% True Color (RGB)** → metode *chromatic artifact* tidak dapat berdiri sendiri; pipeline wajib menggabungkan fitur **Residu Noise & Copy-Paste**.
- **CPI (Copy-Paste di struk yang sama)** mendominasi (`355` area), dan **Total/payment** adalah target utama pemalsuan (`243` area / ~52%).
- Ditemukan 1 anomali pada rilis resmi `findit2.zip`: `X51006619709.png` tercatat sebagai `forged=1` di `val.txt`, tetapi file gambarnya tidak ada di folder `val/` (melainkan ada di `train/` sebagai `forged=0`), sehingga dikecualikan dari `val`.

---
## Bagian 3 — Bedah Step-by-Step Pipeline 8 Tahap PCD (*Before & After* Tiap Tahap)

Di bagian ini kita membedah **satu per satu** fungsi dari 8 tahap pengolahan citra klasik menggunakan contoh struk nyata dari split `train`:
- **`X00016469619.png`** → contoh struk asli bertinta biru pudar dengan header digital hitam pekat (`tan woon yann`), sangat jelas menunjukkan pentingnya **CLAHE** dan perbaikan **Multi-Otsu**.
- **`X00016469622.png`** → contoh struk palsu **`CPI`** (angka `8` disalin dari tanggal `25/12/2018` untuk mengubah total `80.90` menjadi `88.90`).
- **`X51005568889.png`** → contoh struk palsu **`IMI`** (angka `1` diketik ulang di Paint di depan `19.60` menjadi `119.60`, memunculkan *ClearType*).

In [ ]:
# Siapkan 3 contoh citra nyata dari split train/test-demo untuk walkthrough visual
demo_faded_path = str(DATA_ROOT / 'test'  / 'X00016469619.png')   # Struk tinta pudar + header digital
demo_cpi_path   = str(DATA_ROOT / 'train' / 'X00016469622.png')   # Struk palsu CPI (80.90 -> 88.90)
demo_imi_path   = str(DATA_ROOT / 'train' / 'X51005568889.png')   # Struk palsu IMI (19.60 -> 119.60)

img_faded = cv2.imread(demo_faded_path, cv2.IMREAD_COLOR)
img_cpi   = cv2.imread(demo_cpi_path,   cv2.IMREAD_COLOR)
img_imi   = cv2.imread(demo_imi_path,   cv2.IMREAD_COLOR)

print(f"Struk Tinta Pudar : {img_faded.shape[1]}x{img_faded.shape[0]} px")
print(f"Struk Palsu CPI   : {img_cpi.shape[1]}x{img_cpi.shape[0]} px")
print(f"Struk Palsu IMI   : {img_imi.shape[1]}x{img_imi.shape[0]} px")

### Tahap 1 & 2 — Pra-pemrosesan: *Before* (Grayscale Mentah) vs *After* (`CLAHE`)
- **Sebab:** Struk belanja hasil scan sering memiliki tinta sangat pudar dan pencahayaan kertas tidak merata.
- **Teknik:** Konversi ke Grayscale dilanjutkan **CLAHE** (`clipLimit=2.0, tileGridSize=(8,8)`).
- **Akibat:** Kontras lokal antara tinta pudar dan kertas menjadi lebih tegas tanpa memicu noise berlebihan, seperti terlihat pada gambar dan histogram di bawah.

In [ ]:
gray_faded = cv2.cvtColor(img_faded, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_faded = clahe.apply(gray_faded)

# Zoom di area nominal tengah struk pudar agar peningkatan kontras terlihat nyata
H_f, W_f = gray_faded.shape
y1, y2, x1, x2 = int(0.45 * H_f), int(0.82 * H_f), int(0.05 * W_f), int(0.95 * W_f)

fig, axes = plt.subplots(2, 2, figsize=(13, 8), gridspec_kw={'height_ratios': [2.2, 1]})
axes[0, 0].imshow(gray_faded[y1:y2, x1:x2], cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('BEFORE: Grayscale Mentah (Tinta Pudar)')
axes[0, 0].axis('off')

axes[0, 1].imshow(clahe_faded[y1:y2, x1:x2], cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title('AFTER: Grayscale + CLAHE (Tinta Lebih Tegas)')
axes[0, 1].axis('off')

axes[1, 0].hist(gray_faded.ravel(), bins=64, range=(0, 256), color='#64748b')
axes[1, 0].set_title('Histogram Intensitas BEFORE (Menumpuk di Terang)')
axes[1, 0].set_xlabel('Intensitas Piksel (0–255)'); axes[1, 0].set_ylabel('Jumlah Piksel')

axes[1, 1].hist(clahe_faded.ravel(), bins=64, range=(0, 256), color='#2563eb')
axes[1, 1].set_title('Histogram Intensitas AFTER CLAHE (Lebih Tersebar)')
axes[1, 1].set_xlabel('Intensitas Piksel (0–255)'); axes[1, 1].set_ylabel('Jumlah Piksel')

plt.tight_layout()
plt.show()

### Tahap 3 — Ruang Warna: *Before* (RGB Biasa) vs *After* (`Sisa Warna / Chromatic Artifact`)
- **Rumus:** $\hat{p} = (R - m, G - m, B - m)$ dengan $m = (R+G+B)/3$, lalu dihitung magnitudo $\|\hat{p}\|_2$ (Gardella dkk., ICCVW 2025).
- **Sebab & Akibat:** Menghapus komponen abu-abu sehingga **hanya anomali warna** yang menyala. Perhatikan zoom-in pada struk **`IMI`** (`X51005568889.png`): seluruh angka asli (`0`, `19`) gelap total (`0`), sedangkan angka **`1`** hasil ketik ulang di Paint **menyala terang** karena pinggiran *Windows ClearType*!

In [ ]:
rgb_imi = cv2.cvtColor(img_imi, cv2.COLOR_BGR2RGB).astype(np.float32)
chroma_imi = np.linalg.norm(rgb_imi - rgb_imi.mean(axis=2, keepdims=True), axis=2)

# Ambil koordinat area IMI pertama pada X51005568889.png untuk zoom-in
r_imi = df_regions[(df_regions['filename'] == 'X51005568889.png') & (~df_regions['is_original_area'])].iloc[0]
pad_z = 45
zx1, zy1 = max(0, r_imi['x'] - pad_z), max(0, r_imi['y'] - pad_z)
zx2, zy2 = min(img_imi.shape[1], r_imi['x'] + r_imi['width'] + pad_z), min(img_imi.shape[0], r_imi['y'] + r_imi['height'] + pad_z)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
axes[0].imshow(rgb_imi[zy1:zy2, zx1:zx2].astype(np.uint8))
axes[0].set_title('BEFORE: Zoom RGB Asli (Angka "1" Sisipan IMI)')

im1 = axes[1].imshow(chroma_imi[zy1:zy2, zx1:zx2], cmap='inferno', vmin=0, vmax=60)
axes[1].set_title('AFTER: Peta Sisa Warna ||RGB - mean|| (ClearType Menyala!)')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

# Bandingkan rata-rata chroma pinggiran angka '1' palsu vs seluruh area zoom
axes[2].imshow(cv2.cvtColor(img_faded, cv2.COLOR_BGR2RGB)[y1:y2, x1:x2])
chroma_faded = np.linalg.norm(cv2.cvtColor(img_faded, cv2.COLOR_BGR2RGB).astype(np.float32) - cv2.cvtColor(img_faded, cv2.COLOR_BGR2RGB).astype(np.float32).mean(axis=2, keepdims=True), axis=2)
axes[2].imshow(chroma_faded[y1:y2, x1:x2], cmap='inferno', vmin=0, vmax=40)
axes[2].set_title('Kontras di Struk Berwarna: Bias Lensa Scanner & Tinta Biru')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Tahap 4 — Filtering: *Before* (Grayscale) → `Median Filter 3×3` → *After* (`Residu Noise`)
- **Rumus:** $\text{Residu} = I_{\text{gray}} - \text{MedianFilter}_{3\times 3}(I_{\text{gray}})$
- **Sebab & Akibat:** Median filter membuang bintik halus sambil menjaga bentuk utama. Selisih keduanya menghasilkan lapisan tekstur bintik (*noise residual*) yang berfungsi sebagai "sidik jari scanner".

In [ ]:
gray_cpi = cv2.cvtColor(img_cpi, cv2.COLOR_BGR2GRAY)
med_cpi = cv2.medianBlur(gray_cpi, 3)
noise_cpi = gray_cpi.astype(np.float32) - med_cpi.astype(np.float32)

# Zoom di sekitar area 88.91 pada struk CPI X00016469622.png
r_cpi = df_regions[(df_regions['filename'] == 'X00016469622.png') & (~df_regions['is_original_area'])].iloc[0]
cy1, cy2 = max(0, r_cpi['y'] - 35), min(gray_cpi.shape[0], r_cpi['y'] + r_cpi['height'] + 45)
cx1, cx2 = max(0, r_cpi['x'] - 30), min(gray_cpi.shape[1], r_cpi['x'] + r_cpi['width'] + 95)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].imshow(gray_cpi[cy1:cy2, cx1:cx2], cmap='gray', vmin=0, vmax=255)
axes[0].set_title('BEFORE: Citra Grayscale Asli')

axes[1].imshow(med_cpi[cy1:cy2, cx1:cx2], cmap='gray', vmin=0, vmax=255)
axes[1].set_title('PROSES: Hasil Median Filter 3×3 (Lebih Halus)')

im_n = axes[2].imshow(np.abs(noise_cpi[cy1:cy2, cx1:cx2]), cmap='magma', vmin=0, vmax=25)
axes[2].set_title('AFTER: Residu Noise |Gray - Median|')
plt.colorbar(im_n, ax=axes[2], fraction=0.046, pad=0.04)

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Tahap 5 — Segmentasi: *Before* (Multi-Otsu `thresholds[0]` — Gagal!) vs *After* (`thresholds[1]` + `Adaptive Gaussian`)
- **Sebab:** Struk pudar yang punya header digital hitam pekat (`tan woon yann`) memiliki **3 kelas kecerahan** (header pekat, tinta struk pudar, kertas putih).
- **Akibat:** Jika memakai batas bawah (`thresholds[0]`), 97% teks struk hilang dan hanya tersisa `tan woon yann`. Dengan memakai batas atas (`thresholds[1]`) di-`AND`-kan dengan `Adaptive Threshold`, seluruh 400+ karakter struk tersegmentasi bersih!

In [ ]:
th = threshold_multiotsu(clahe_faded, classes=3)
mask_before = (clahe_faded < th[0]).astype(np.uint8) * 255
mask_otsu_t1 = (clahe_faded < th[1]).astype(np.uint8) * 255
mask_adapt = cv2.adaptiveThreshold(clahe_faded, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 25, 10)
mask_after = cv2.bitwise_and(mask_otsu_t1, mask_adapt)

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
axes[0].hist(clahe_faded.ravel(), bins=64, range=(0, 256), color='#94a3b8')
axes[0].axvline(th[0], color='#dc2626', linestyle='--', linewidth=2, label=f'th[0] = {th[0]} (terlalu gelap)')
axes[0].axvline(th[1], color='#16a34a', linestyle='-',  linewidth=2, label=f'th[1] = {th[1]} (dipakai)')
axes[0].set_title('Posisi 2 Batas Multi-Otsu (3 Kelas)')
axes[0].set_xlabel('Intensitas CLAHE'); axes[0].legend()

axes[1].imshow(mask_before, cmap='gray')
axes[1].set_title('BEFORE (th[0]): Teks Struk Hilang Total!')
axes[1].axis('off')

axes[2].imshow(mask_after, cmap='gray')
axes[2].set_title('AFTER (th[1] & Adaptive): Seluruh Teks Terbaca!')
axes[2].axis('off')

plt.tight_layout()
plt.show()

### Tahap 6 — Morfologi & Connected Components: Pembersihan Debu, Pemotongan Per Karakter, & Penggabungan Blok Kata
- **Opening (`2×2`)** membuang bintik debu tanpa mengecilkan huruf.
- **Connected Components #1** memberi ID unik dan kotak `(x, y, w, h)` pada setiap huruf/angka (ditampilkan warna-warni di panel 2).
- **Dilasi Horizontal + Connected Components #2** menggabungkan karakter berdekatan menjadi satu blok kata/nominal (pengganti detektor teks neural network `EAST`).

In [ ]:
k_open = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
clean_mask = cv2.morphologyEx(mask_after, cv2.MORPH_OPEN, k_open)
n_cc, labels_cc, stats_cc, _ = cv2.connectedComponentsWithStats(clean_mask, connectivity=8)

# Visualisasi berwarna untuk Connected Components (tiap karakter 1 warna unik)
rng = np.random.default_rng(42)
palette_cc = rng.integers(50, 255, size=(n_cc, 3), dtype=np.uint8)
palette_cc[0] = [0, 0, 0]
cc_vis = palette_cc[labels_cc]

valid_w = [stats_cc[i, cv2.CC_STAT_WIDTH] for i in range(1, n_cc) if 20 <= stats_cc[i, cv2.CC_STAT_AREA] <= 8000]
med_w = int(np.clip(np.median(valid_w) * 0.9, 5, 22)) if valid_w else 10
word_mask = cv2.dilate(clean_mask, cv2.getStructuringElement(cv2.MORPH_RECT, (med_w, 3)), iterations=1)
n_words, _, w_stats, _ = cv2.connectedComponentsWithStats(word_mask, connectivity=8)

# Gambar kotak hijau blok kata hasil Dilasi Horizontal di atas citra struk
word_overlay = cv2.cvtColor(img_faded, cv2.COLOR_BGR2RGB).copy()
for wid in range(1, n_words):
    wx, wy, ww, wh, wa = w_stats[wid]
    if wa >= 40:
        cv2.rectangle(word_overlay, (wx, wy), (wx + ww, wy + wh), (22, 163, 74), 2)

fig, axes = plt.subplots(1, 3, figsize=(15, 6.5))
axes[0].imshow(clean_mask, cmap='gray')
axes[0].set_title(f'1. After Opening (Bersih dari Debu)')

axes[1].imshow(cc_vis)
axes[1].set_title(f'2. Connected Components #1 ({len(valid_w)} Karakter Unik)')

axes[2].imshow(word_overlay)
axes[2].set_title(f'3. Dilasi Horizontal + CC #2 (Pengganti EAST)')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Tahap 7 — Bukti Nyata Forensik `CPI`: Angka *Copy-Paste* (`MAD ≈ 0`) vs Angka Kembar Asli (`MAD Tinggi`)

Inilah inti pembeda sistem kita! Pada citra `X00016469622.png`, pelaku menyalin angka **`8`** dari tanggal `25/12/2018` (`ORIG`) lalu menempelkannya ke harga `88.91` (`CPI`).
- Saat kita kurangkan piksel **Sumber vs Salinan (`|P1 - P2|`)**, hasilnya nyaris **hitam pekat (`MAD ≈ 0`, hampir 100% piksel identik)** karena disimpan *lossless PNG*.
- Sebaliknya, pada dua angka **`0` asli** yang dicetak alami di struk yang sama, peta selisihnya **terang (`MAD tinggi`)** karena ada perbedaan tinta, posisi sub-piksel, dan noise scanner!

In [ ]:
# Ambil pasangan karakter CPI paling identik hasil Connected Components pada struk palsu X00016469622.png
# (kotak anotasi manual VIA sering agak longgar 1-3 px, sehingga kita pakai kotak pas dari CC seperti di pipeline)
clahe_cpi = clahe.apply(gray_cpi)
th_c = threshold_multiotsu(clahe_cpi, classes=3)
m_c = cv2.bitwise_and(
    (clahe_cpi < th_c[1]).astype(np.uint8) * 255,
    cv2.adaptiveThreshold(clahe_cpi, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 25, 10)
)
m_c = cv2.morphologyEx(m_c, cv2.MORPH_OPEN, k_open)
nc_cc, _, sc_cc, _ = cv2.connectedComponentsWithStats(m_c, connectivity=8)
gray_cpi_f = gray_cpi.astype(np.float32)

# Cari pasangan CC di area isi struk (y > 150) yang paling identik (bukti CPI nyata)
cands_c = [sc_cc[i] for i in range(1, nc_cc) if sc_cc[i, 1] > 150 and sc_cc[i, 2] >= 6 and sc_cc[i, 3] >= 10 and sc_cc[i, 4] >= 30]
best_mad_cp, p_orig, best_p_copy = 999.0, None, None
pad_p = 3
for i in range(len(cands_c)):
    x1, y1, w1, h1, _ = cands_c[i]
    for j in range(i + 1, len(cands_c)):
        x2, y2, w2, h2, _ = cands_c[j]
        if w1 == w2 and h1 == h2 and np.hypot(x1 - x2, y1 - y2) > 25:
            q1 = gray_cpi_f[max(0, y1-pad_p):y1+h1+pad_p, max(0, x1-pad_p):x1+w1+pad_p]
            q2 = gray_cpi_f[max(0, y2-pad_p):y2+h2+pad_p, max(0, x2-pad_p):x2+w2+pad_p]
            if q1.shape != q2.shape:
                continue
            nw = (q1 < 248) | (q2 < 248)
            if nw.sum() < 30:
                continue
            m_val = float(np.abs(q1[nw] - q2[nw]).mean())
            if m_val < best_mad_cp:
                best_mad_cp, p_orig, best_p_copy = m_val, q1, q2

diff_cp = np.abs(p_orig - best_p_copy)
nw_cp = (p_orig < 248) | (best_p_copy < 248)
mad_cp_nw = float(diff_cp[nw_cp].mean())
exact_cp_nw = float((diff_cp[nw_cp] <= 2.0).mean()) * 100

# Pembanding: 2 karakter cetak asli berukuran sama pada struk asli X00016469619.png
gray_asli = cv2.cvtColor(img_faded, cv2.COLOR_BGR2GRAY).astype(np.float32)
th_a = threshold_multiotsu(clahe_faded, classes=3)
m_a = cv2.bitwise_and((clahe_faded < th_a[1]).astype(np.uint8) * 255, mask_adapt)
m_a = cv2.morphologyEx(m_a, cv2.MORPH_OPEN, k_open)
na_cc, _, sa_cc, _ = cv2.connectedComponentsWithStats(m_a, connectivity=8)
cands_a = [sa_cc[i] for i in range(1, na_cc) if sa_cc[i, 1] > 200 and 7 <= sa_cc[i, 2] <= 14 and 13 <= sa_cc[i, 3] <= 20]
pa1, pa2, best_a_mad = None, None, 999.0
for i in range(min(len(cands_a), 60)):
    for j in range(i + 1, min(len(cands_a), 60)):
        x1, y1, w1, h1, _ = cands_a[i]
        x2, y2, w2, h2, _ = cands_a[j]
        if w1 == w2 and h1 == h2 and np.hypot(x1 - x2, y1 - y2) > 40:
            q1 = gray_asli[max(0, y1-pad_p):y1+h1+pad_p, max(0, x1-pad_p):x1+w1+pad_p]
            q2 = gray_asli[max(0, y2-pad_p):y2+h2+pad_p, max(0, x2-pad_p):x2+w2+pad_p]
            if q1.shape != q2.shape:
                continue
            nw = (q1 < 248) | (q2 < 248)
            if nw.sum() < 30:
                continue
            m_val = float(np.abs(q1[nw] - q2[nw]).mean())
            if 8.0 < m_val < best_a_mad:
                best_a_mad, pa1, pa2 = m_val, q1, q2

diff_asli = np.abs(pa1 - pa2)
nw_a = (pa1 < 248) | (pa2 < 248)
mad_a_nw = float(diff_asli[nw_a].mean())
exact_a_nw = float((diff_asli[nw_a] <= 2.0).mean()) * 100

fig, axes = plt.subplots(2, 3, figsize=(12, 7.5))
axes[0, 0].imshow(p_orig, cmap='gray', vmin=0, vmax=255); axes[0, 0].set_title('PALSU CPI: Karakter #1 (Sumber/Salinan)')
axes[0, 1].imshow(best_p_copy, cmap='gray', vmin=0, vmax=255); axes[0, 1].set_title('PALSU CPI: Karakter #2 (Salinan)')
im0 = axes[0, 2].imshow(diff_cp, cmap='hot', vmin=0, vmax=60)
axes[0, 2].set_title(f'Selisih |P1 - P2| (Nyaris Hitam!)\nMAD={mad_cp_nw:.2f} | Identik={exact_cp_nw:.1f}%')
plt.colorbar(im0, ax=axes[0, 2], fraction=0.046)

axes[1, 0].imshow(pa1, cmap='gray', vmin=0, vmax=255); axes[1, 0].set_title('ASLI: Karakter Cetak #1')
axes[1, 1].imshow(pa2, cmap='gray', vmin=0, vmax=255); axes[1, 1].set_title('ASLI: Karakter Cetak #2 (Kembar)')
im1 = axes[1, 2].imshow(diff_asli, cmap='hot', vmin=0, vmax=60)
axes[1, 2].set_title(f'Selisih |P1 - P2| (Banyak Beda!)\nMAD={mad_a_nw:.2f} | Identik={exact_a_nw:.1f}%')
plt.colorbar(im1, ax=axes[1, 2], fraction=0.046)

for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()

---
## Bagian 4 — Pipeline Lengkap Modular & Tuning Threshold di Split `val` (192 Struk)

Sekarang kita gabungkan seluruh 8 tahap di atas ke dalam dua fungsi bersih (`extract_receipt_forensics` dan `predict_receipt`) yang juga akan dipakai identik oleh aplikasi **Streamlit**. Untuk menghemat waktu jika notebook dijalankan ulang, fitur hasil ekstraksi otomatis disimpan sebagai cache JSON di Google Drive (`USE_CACHE = True`).

In [ ]:
def extract_receipt_forensics(filepath: str, min_area: int = 20, max_area: int = 8000, pad: int = 3) -> dict:
    """Menjalankan Pipeline OpenCV Klasik Tahap 1–7 pada 1 citra struk."""
    img_bgr = cv2.imread(filepath, cv2.IMREAD_COLOR)
    if img_bgr is None:
        return {'filepath': filepath, 'error': True, 'cpi_candidates': [], 'imi_cleartype': [], 'imi_nfa': []}

    H, W = img_bgr.shape[:2]

    # Tahap 2: Grayscale + CLAHE
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray_clahe = clahe.apply(gray)

    # Tahap 3: Ruang warna (Sisa warna / Chromatic residual)
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    chroma_mag = np.linalg.norm(rgb - rgb.mean(axis=2, keepdims=True), axis=2)
    is_color = float(chroma_mag.max()) > 1e-3

    # Tahap 4: Filtering (Residu noise = gray - medianBlur 3x3)
    noise_res = gray.astype(np.float32) - cv2.medianBlur(gray, 3).astype(np.float32)

    # Tahap 5: Segmentasi (Multi-Otsu thresholds[1] + Adaptive Gaussian)
    try:
        thresholds = threshold_multiotsu(gray_clahe, classes=3)
        otsu_mask = (gray_clahe < thresholds[1]).astype(np.uint8) * 255
    except Exception:
        _, otsu_mask = cv2.threshold(gray_clahe, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    adapt_mask = cv2.adaptiveThreshold(
        gray_clahe, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 25, 10
    )
    ink_mask = cv2.bitwise_and(otsu_mask, adapt_mask)

    # Tahap 6a: Morfologi Opening + masker cincin tepi 1-px
    k_open = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    clean_mask = cv2.morphologyEx(ink_mask, cv2.MORPH_OPEN, k_open)
    ink_dil1 = cv2.dilate(clean_mask, np.ones((3, 3), np.uint8), iterations=1)
    fringe_ring = cv2.subtract(ink_dil1, clean_mask) > 0

    # Connected Components #1 (per karakter)
    num_labels, labels, stats_cc, _ = cv2.connectedComponentsWithStats(clean_mask, connectivity=8)
    chars = []
    gray_f = gray.astype(np.float32)

    for cid in range(1, num_labels):
        x, y, w, h, area = stats_cc[cid]
        if area < min_area or area > max_area:
            continue
        if h < 7 or w < 3 or (w / max(h, 1)) > 4.5 or (h / max(w, 1)) > 8.0:
            continue

        c_mask = (labels[y:y+h, x:x+w] == cid)
        g_core = gray_f[y:y+h, x:x+w][c_mask]
        n_core = noise_res[y:y+h, x:x+w][c_mask]
        ch_core = chroma_mag[y:y+h, x:x+w][c_mask]

        px1, py1 = max(0, x - pad), max(0, y - pad)
        px2, py2 = min(W, x + w + pad), min(H, y + h + pad)
        g_pad = gray_f[py1:py2, px1:px2]
        n_pad = noise_res[py1:py2, px1:px2]
        bg_pad = (ink_dil1[py1:py2, px1:px2] == 0)
        fr_mask = fringe_ring[py1:py2, px1:px2]

        core_chroma_mean = float(ch_core.mean()) if is_color else 0.0
        fringe_chroma_mean = float(chroma_mag[py1:py2, px1:px2][fr_mask].mean()) if (is_color and fr_mask.any()) else 0.0

        chars.append({
            'bbox': [int(x), int(y), int(w), int(h)],
            'area': int(area),
            'noise_std': float(n_core.std()),
            'chroma_std': float(ch_core.std()) if is_color else 0.0,
            'core_chroma': core_chroma_mean,
            'cleartype_diff': max(0.0, fringe_chroma_mean - core_chroma_mean),
            'g_pad': g_pad, 'n_pad': n_pad, 'bg_pad': bg_pad,
            'in_top_header': (y < 0.10 * H)
        })

    # Tahap 6b: Morfologi Dilasi Horizontal + CC #2 (blok kata/nominal)
    med_w = int(np.clip(np.median([c['bbox'][2] for c in chars]) * 0.9, 5, 22)) if chars else 10
    word_mask = cv2.dilate(clean_mask, cv2.getStructuringElement(cv2.MORPH_RECT, (med_w, 3)), iterations=1)
    w_num, _, w_stats, _ = cv2.connectedComponentsWithStats(word_mask, connectivity=8)
    words = []
    for wid in range(1, w_num):
        wx, wy, ww, wh, warea = w_stats[wid]
        if warea < min_area * 2:
            continue
        m_idx = [
            idx for idx, c in enumerate(chars)
            if wx <= (c['bbox'][0] + c['bbox'][2] / 2) <= wx + ww
            and wy <= (c['bbox'][1] + c['bbox'][3] / 2) <= wy + wh
        ]
        if m_idx:
            words.append({'bbox': [int(wx), int(wy), int(ww), int(wh)], 'char_indices': m_idx})

    # Tahap 7A: Cabang CPI (Size-Bucketed Exact/Near-Exact MAD + NCC)
    buckets = defaultdict(list)
    for idx, c in enumerate(chars):
        x, y, w, h = c['bbox']
        if w < 6 or h < 10 or c['area'] < 30 or c['g_pad'].std() < 15.0:
            continue
        buckets[(w, h)].append(idx)

    cpi_candidates = []
    visited_pairs = set()
    for (bw, bh), idxs in buckets.items():
        cand_j = [k for dw in (-1, 0, 1) for dh in (-1, 0, 1) for k in buckets.get((bw + dw, bh + dh), [])]
        for i in idxs:
            ci = chars[i]
            x1, y1, w1, h1 = ci['bbox']
            g1, n1, bg1 = ci['g_pad'], ci['n_pad'], ci['bg_pad']
            h1p, w1p = g1.shape
            for j in cand_j:
                if j <= i or (i, j) in visited_pairs:
                    continue
                visited_pairs.add((i, j))
                cj = chars[j]
                x2, y2, w2, h2 = cj['bbox']
                dist = float(np.hypot(x1 - x2, y1 - y2))
                if dist < 20.0:
                    continue
                g2, n2, bg2 = cj['g_pad'], cj['n_pad'], cj['bg_pad']
                ch_min, cw_min = min(h1p, g2.shape[0]), min(w1p, g2.shape[1])
                if ch_min < 12 or cw_min < 8:
                    continue
                o1y, o1x = (h1p - ch_min) // 2, (w1p - cw_min) // 2
                o2y, o2x = (g2.shape[0] - ch_min) // 2, (g2.shape[1] - cw_min) // 2
                p1 = g1[o1y:o1y+ch_min, o1x:o1x+cw_min]
                p2 = g2[o2y:o2y+ch_min, o2x:o2x+cw_min]

                nw_mask = (p1 < 248.0) | (p2 < 248.0)
                n_nw = int(nw_mask.sum())
                if n_nw < 25:
                    continue
                abs_diff = np.abs(p1[nw_mask] - p2[nw_mask])
                mad_nw = float(abs_diff.mean())
                if mad_nw > 18.0:
                    continue
                exact_nw = float((abs_diff <= 2.0).mean())

                p1z, p2z = p1 - p1.mean(), p2 - p2.mean()
                den_raw = float(np.linalg.norm(p1z) * np.linalg.norm(p2z))
                ncc_raw = float(np.sum(p1z * p2z) / den_raw) if den_raw > 1e-4 else 0.0
                if ncc_raw < 0.85:
                    continue

                r1 = n1[o1y:o1y+ch_min, o1x:o1x+cw_min]
                r2 = n2[o2y:o2y+ch_min, o2x:o2x+cw_min]
                cbg = bg1[o1y:o1y+ch_min, o1x:o1x+cw_min] & bg2[o2y:o2y+ch_min, o2x:o2x+cw_min]
                if int(cbg.sum()) >= 15:
                    v1, v2 = r1[cbg] - r1[cbg].mean(), r2[cbg] - r2[cbg].mean()
                    den_h = float(np.linalg.norm(v1) * np.linalg.norm(v2))
                    ncc_halo = float(np.dot(v1, v2) / den_h) if den_h > 1e-4 else 0.0
                else:
                    ncc_halo = 0.0

                cpi_candidates.append({
                    'bbox1': ci['bbox'], 'bbox2': cj['bbox'],
                    'mad_nw': round(mad_nw, 3), 'exact_nw': round(exact_nw, 4),
                    'ncc_raw': round(ncc_raw, 4), 'ncc_halo': round(ncc_halo, 4),
                    'dist': round(dist, 1), 'n_nw': n_nw,
                    'both_header': bool(ci['in_top_header'] and cj['in_top_header']),
                    'same_line_close': bool(abs(y1 - y2) <= 4 and dist < 50.0)
                })

    cpi_candidates.sort(key=lambda d: (d['mad_nw'], -d['exact_nw']))
    cpi_candidates = cpi_candidates[:40]

    # Tahap 7B: Cabang IMI (ClearType Fringe Anomaly + Gardella NFA)
    imi_cleartype, imi_nfa = [], []
    if len(chars) >= 15:
        if is_color:
            ct_vals = np.array([c['cleartype_diff'] for c in chars], dtype=np.float64)
            med_ct = float(np.median(ct_vals))
            mad_ct = float(np.median(np.abs(ct_vals - med_ct))) + 0.25
            if med_ct < 2.5:
                for c, v in zip(chars, ct_vals):
                    if c['in_top_header']:
                        continue
                    if c['core_chroma'] < 12.0 and v >= 6.0:
                        rz = (v - med_ct) / (1.4826 * mad_ct)
                        if rz >= 6.0:
                            imi_cleartype.append({'bbox': c['bbox'], 'ct_diff': round(float(v), 2), 'core_chroma': round(c['core_chroma'], 2), 'robust_z': round(float(rz), 2)})
                imi_cleartype.sort(key=lambda d: d['robust_z'], reverse=True)
                imi_cleartype = imi_cleartype[:20]

        alpha = 0.10
        key = 'chroma_std' if (is_color and np.median([c['chroma_std'] for c in chars]) > 1.5) else 'noise_std'
        vals = np.array([c[key] for c in chars], dtype=np.float64)
        mu, sigma = vals.mean(), vals.std(ddof=1) + 1e-8
        is_out = vals < (mu - stats.norm.ppf(1.0 - alpha) * sigma)
        n_tests = max(len(words), 1)
        for w in words:
            idxs = w['char_indices']
            if len(idxs) < 3 or w['bbox'][1] < 0.10 * H:
                continue
            k = int(is_out[idxs].sum())
            if k >= 2:
                nfa = n_tests * float(stats.binom.sf(k - 1, len(idxs), alpha))
                if nfa < 0.05:
                    imi_nfa.append({'bbox': w['bbox'], 'n_chars': len(idxs), 'k': k, 'nfa': float(nfa), 'feat': key})
        imi_nfa.sort(key=lambda d: d['nfa'])
        imi_nfa = imi_nfa[:10]

    return {
        'filepath': filepath, 'shape': [H, W], 'is_color': is_color,
        'n_chars': len(chars), 'n_words': len(words),
        'cpi_candidates': cpi_candidates, 'imi_cleartype': imi_cleartype, 'imi_nfa': imi_nfa
    }


def predict_receipt(res: dict, cfg: dict) -> dict:
    """Tahap 8: Mengambil keputusan Asli/Palsu dan kotak area curiga berdasarkan threshold terkunci."""
    cpi_boxes, cpi_pairs = [], []
    for c in res['cpi_candidates']:
        if cfg.get('ignore_top_header', True) and c['both_header']:
            continue
        if cfg.get('ignore_same_line_close', True) and c['same_line_close']:
            continue
        if (
            c['mad_nw'] <= cfg['mad_max']
            and c['exact_nw'] >= cfg['exact_min']
            and c['ncc_raw'] >= cfg['ncc_raw_min']
            and c['n_nw'] >= cfg['min_nw_pixels']
        ):
            cpi_pairs.append(c)
            cpi_boxes.append({'bbox': c['bbox1'], 'reason': f"CPI(MAD={c['mad_nw']})", 'score': c['exact_nw']})
            cpi_boxes.append({'bbox': c['bbox2'], 'reason': f"CPI(MAD={c['mad_nw']})", 'score': c['exact_nw']})

    imi_boxes = []
    ct_hits = [h for h in res['imi_cleartype'] if h['robust_z'] >= cfg['ct_z_min'] and h['ct_diff'] >= cfg['ct_diff_min']]
    if len(ct_hits) >= cfg['ct_min_chars']:
        for h in ct_hits:
            imi_boxes.append({'bbox': h['bbox'], 'reason': f"IMI_CT(z={h['robust_z']})", 'score': h['robust_z']})

    if cfg.get('use_nfa', False):
        for h in res['imi_nfa']:
            if h['nfa'] < cfg['nfa_max']:
                imi_boxes.append({'bbox': h['bbox'], 'reason': f"IMI_NFA({h['nfa']:.1e})", 'score': -np.log10(h['nfa'] + 1e-12)})

    all_boxes = cpi_boxes + imi_boxes
    return {'pred': 1 if len(all_boxes) > 0 else 0, 'boxes': all_boxes, 'cpi_pairs': cpi_pairs, 'n_cpi': len(cpi_pairs), 'n_imi': len(imi_boxes)}


def compute_cls_metrics(y_true, y_pred) -> dict:
    y_true, y_pred = np.asarray(y_true, dtype=int), np.asarray(y_pred, dtype=int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn, 'Precision': prec, 'Recall': rec, 'F1': f1, 'Accuracy': (tp + tn) / len(y_true)}

print("Fungsi pipeline modular berhasil didefinisikan.")

In [ ]:
# Ekstraksi fitur pada VAL (192 struk) dengan dukungan cache Drive agar re-run cepat
USE_CACHE = True
VAL_CACHE = PROJECT_DRIVE_DIR / 'cache_val_features_final.json'

if USE_CACHE and VAL_CACHE.exists():
    print(f"Memuat fitur VAL dari cache: {VAL_CACHE}")
    val_results = json.loads(VAL_CACHE.read_text(encoding='utf-8'))
else:
    t0 = time.time()
    print(f"Mengekstrak fitur pada {len(df_val)} struk VAL (CPU paralel)...")
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 2) as ex:
        val_results = list(ex.map(extract_receipt_forensics, df_val['filepath'].tolist()))
    print(f"Ekstraksi VAL selesai dalam {time.time() - t0:.1f} detik.")
    VAL_CACHE.write_text(json.dumps(val_results), encoding='utf-8')
    print(f"Cache VAL disimpan ke {VAL_CACHE}")

# Grid Search threshold HANYA pada split VAL
y_val = df_val['forged'].values
best_score, best_cfg, best_val_metrics = -1.0, None, None
grid_logs = []

for mad_max in [0.5, 1.0, 2.0, 3.0, 4.5, 6.0]:
    for exact_min in [0.65, 0.75, 0.85, 0.92, 0.97]:
        for min_nw in [30, 50, 75]:
            for ign_sl in [True, False]:
                for ct_z in [12.0, 20.0, 999.0]:
                    cfg = {
                        'mad_max': mad_max, 'exact_min': exact_min, 'ncc_raw_min': 0.92,
                        'min_nw_pixels': min_nw, 'ignore_top_header': True,
                        'ignore_same_line_close': ign_sl, 'ct_z_min': ct_z,
                        'ct_diff_min': 10.0, 'ct_min_chars': 2, 'use_nfa': False, 'nfa_max': 0.001
                    }
                    m = compute_cls_metrics(y_val, [predict_receipt(r, cfg)['pred'] for r in val_results])
                    if m['F1'] + 0.01 * m['Precision'] > best_score:
                        best_score, best_cfg, best_val_metrics = m['F1'] + 0.01 * m['Precision'], cfg.copy(), m.copy()
                    grid_logs.append({**cfg, **m})

# Visualisasi kurva sensitivitas threshold MAD pada VAL
mad_curve = []
for mm in np.linspace(0.5, 10.0, 20):
    c_tmp = {**best_cfg, 'mad_max': float(mm)}
    m_tmp = compute_cls_metrics(y_val, [predict_receipt(r, c_tmp)['pred'] for r in val_results])
    mad_curve.append({'mad_max': mm, 'Precision': m_tmp['Precision']*100, 'Recall': m_tmp['Recall']*100, 'F1': m_tmp['F1']*100})
df_curve = pd.DataFrame(mad_curve)

plt.figure(figsize=(8.5, 4.2))
plt.plot(df_curve['mad_max'], df_curve['Precision'], marker='o', label='Precision (%)', color='#2563eb')
plt.plot(df_curve['mad_max'], df_curve['Recall'],    marker='s', label='Recall (%)',    color='#f59e0b')
plt.plot(df_curve['mad_max'], df_curve['F1'],        marker='^', label='F1-Score (%)',  color='#dc2626', linewidth=2.5)
plt.axvline(best_cfg['mad_max'], color='#16a34a', linestyle='--', label=f"Threshold Terpilih (MAD <= {best_cfg['mad_max']})")
plt.title('Kurva Tuning Threshold MAD pada Split VAL (192 Struk)')
plt.xlabel('Batas Maksimum MAD Piksel Non-Putih'); plt.ylabel('Persentase (%)')
plt.ylim(0, 105); plt.grid(True, linestyle='--', alpha=0.4); plt.legend()
plt.tight_layout(); plt.show()

print(f"Skor VAL Terbaik -> Precision: {best_val_metrics['Precision']*100:.2f}% | Recall: {best_val_metrics['Recall']*100:.2f}% | F1: {best_val_metrics['F1']*100:.2f}%")
print("Konfigurasi Terkunci:", json.dumps(best_cfg, indent=2))

---
## Bagian 5 — Evaluasi Resmi pada Split `test` (218 Struk, 35 Palsu)

Konfigurasi terbaik dari `val` dikunci tanpa diubah-ubah, lalu dijalankan sekali pada 218 struk uji untuk memperoleh skor final yang dapat dibandingkan secara langsung dengan baseline paper ICDAR 2023.

In [ ]:
TEST_CACHE = PROJECT_DRIVE_DIR / 'cache_test_features_final.json'
if USE_CACHE and TEST_CACHE.exists():
    print(f"Memuat fitur TEST dari cache: {TEST_CACHE}")
    test_results = json.loads(TEST_CACHE.read_text(encoding='utf-8'))
else:
    t0 = time.time()
    print(f"Mengekstrak fitur pada {len(df_test)} struk TEST (CPU paralel)...")
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 2) as ex:
        test_results = list(ex.map(extract_receipt_forensics, df_test['filepath'].tolist()))
    print(f"Ekstraksi TEST selesai dalam {time.time() - t0:.1f} detik.")
    TEST_CACHE.write_text(json.dumps(test_results), encoding='utf-8')
    print(f"Cache TEST disimpan ke {TEST_CACHE}")

test_preds_obj = [predict_receipt(r, best_cfg) for r in test_results]
df_test_eval = df_test.copy()
df_test_eval['pred'] = [p['pred'] for p in test_preds_obj]
df_test_eval['n_cpi_pairs'] = [p['n_cpi'] for p in test_preds_obj]
df_test_eval['n_imi_boxes'] = [p['n_imi'] for p in test_preds_obj]
m_test = compute_cls_metrics(df_test_eval['forged'].values, df_test_eval['pred'].values)

bench_df = pd.DataFrame([
    {'Metode': 'SVM Citra (ICDAR 2023)',            'Precision (%)': 30.00, 'Recall (%)':  8.57, 'F1 (%)': 13.33},
    {'Metode': 'BoW + LR Teks (ICDAR 2023)',        'Precision (%)': 40.00, 'Recall (%)': 11.43, 'F1 (%)': 17.78},
    {'Metode': 'ChatGPT Strict (ICDAR 2023)',       'Precision (%)': 14.69, 'Recall (%)': 88.57, 'F1 (%)': 25.20},
    {'Metode': 'ChatGPT Relaxed (ICDAR 2023)',      'Precision (%)': 18.33, 'Recall (%)': 62.86, 'F1 (%)': 28.39},
    {'Metode': 'OpenCV Klasik Final (Kelompok 4)',  'Precision (%)': round(m_test['Precision']*100, 2),
                                                    'Recall (%)': round(m_test['Recall']*100, 2),
                                                    'F1 (%)': round(m_test['F1']*100, 2)},
])
print(f"=== HASIL FINAL SPLIT TEST: Precision={m_test['Precision']*100:.2f}% | Recall={m_test['Recall']*100:.2f}% | F1={m_test['F1']*100:.2f}% ===")
display(bench_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].barh(bench_df['Metode'], bench_df['F1 (%)'], color=['#64748b']*4 + ['#dc2626'])
axes[0].set_title('Perbandingan F1 Kelas Palsu di Split TEST (218 Struk)')
axes[0].set_xlabel('F1-Score (%)'); axes[0].set_xlim(0, 100)
for i, v in enumerate(bench_df['F1 (%)']):
    axes[0].text(v + 1.5, i, f"{v:.2f}%", va='center', fontweight='bold')

cm = np.array([[m_test['TN'], m_test['FP']], [m_test['FN'], m_test['TP']]])
axes[1].imshow(cm, cmap='Blues')
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['Pred Asli', 'Pred Palsu'])
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(['Aktual Asli', 'Aktual Palsu'])
axes[1].set_title('Confusion Matrix Split TEST')
for r in range(2):
    for c in range(2):
        axes[1].text(c, r, str(cm[r, c]), ha='center', va='center', fontsize=15, fontweight='bold',
                     color='white' if cm[r, c] > cm.max()/2 else 'black')
plt.tight_layout(); plt.show()

In [ ]:
# Evaluasi Tingkat Area (IoU), Per Teknik, Per Jenis Isi (Total/Payment), dan Audit False Positive
def calc_iou(b1, b2):
    x1, y1, w1, h1 = b1
    x2, y2, w2, h2 = b2
    inter = max(0, min(x1+w1, x2+w2) - max(x1, x2)) * max(0, min(y1+h1, y2+h2) - max(y1, y2))
    return inter / max(w1*h1 + w2*h2 - inter, 1e-8) if inter > 0 else 0.0

df_test_mod = df_regions[(df_regions['split'] == 'test') & (~df_regions['is_original_area'])].copy()
pred_obj_map = {row['filename']: p for (_, row), p in zip(df_test_eval.iterrows(), test_preds_obj)}
pred_lbl_map = dict(zip(df_test_eval['filename'], df_test_eval['pred']))

df_test_mod['best_iou'] = [
    max((calc_iou((r['x'], r['y'], r['width'], r['height']), b['bbox']) for b in pred_obj_map[r['filename']]['boxes']), default=0.0)
    for _, r in df_test_mod.iterrows()
]

print(f"Mean Best IoU (101 Modified Area) : {df_test_mod['best_iou'].mean():.4f}")
print(f"Area Overlap IoU >= 0.10          : {(df_test_mod['best_iou'] >= 0.10).mean()*100:.2f}% ({(df_test_mod['best_iou'] >= 0.10).sum()}/{len(df_test_mod)})")

tech_rows = []
for tech in ['CPI', 'IMI', 'CUT', 'PIX', 'CPO']:
    sub = df_test_mod[df_test_mod['mod_types'].apply(lambda lst: tech in lst)]
    if len(sub) > 0:
        fnames = sub['filename'].unique()
        tech_rows.append({
            'Teknik': tech, 'Struk': len(fnames), 'Area': len(sub),
            'Recall Struk (%)': round(np.mean([pred_lbl_map[f] for f in fnames])*100, 2),
            'Mean IoU': round(float(sub['best_iou'].mean()), 4),
            'Overlap IoU>=0.1 (%)': round(float((sub['best_iou'] >= 0.10).mean()*100), 2)
        })

ent_rows = []
for ent, sub in df_test_mod.groupby('entity_type'):
    fnames = sub['filename'].unique()
    ent_rows.append({
        'Jenis Isi': ent, 'Struk': len(fnames), 'Area': len(sub),
        'Recall Struk (%)': round(np.mean([pred_lbl_map[f] for f in fnames])*100, 2),
        'Mean IoU': round(float(sub['best_iou'].mean()), 4)
    })

df_tech_res = pd.DataFrame(tech_rows)
df_ent_res = pd.DataFrame(ent_rows).sort_values('Area', ascending=False)
display(df_tech_res)
display(df_ent_res)

# Plot ringkasan Recall per Teknik dan per Jenis Isi
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].bar(df_tech_res['Teknik'], df_tech_res['Recall Struk (%)'], color=['#dc2626', '#f59e0b', '#64748b', '#64748b', '#94a3b8'])
axes[0].set_title('Recall Tingkat Struk per Teknik Pemalsuan (TEST)')
axes[0].set_ylabel('Recall (%)'); axes[0].set_ylim(0, 110)
for i, v in enumerate(df_tech_res['Recall Struk (%)']):
    axes[0].text(i, v + 3, f"{v:.1f}%", ha='center', fontweight='bold')

axes[1].bar(df_ent_res['Jenis Isi'], df_ent_res['Recall Struk (%)'], color=['#dc2626' if 'total' in s.lower() else '#2563eb' for s in df_ent_res['Jenis Isi']])
axes[1].set_title('Recall Tingkat Struk per Jenis Isi (Total/Payment 92,9%!)')
axes[1].set_ylabel('Recall (%)'); axes[1].set_ylim(0, 110)
for i, v in enumerate(df_ent_res['Recall Struk (%)']):
    axes[1].text(i, v + 3, f"{v:.1f}%", ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

# Audit False Positive pada struk Asli
df_asli_t = df_test_eval[df_test_eval['forged'] == 0]
fp_rows = []
for lbl, cond in [
    ('Semua Struk Asli (TEST)', pd.Series(True, index=df_asli_t.index)),
    ('Asli dgn Coretan/Stempel Manual', df_asli_t['handwritten_annotation'] == 1),
    ('Asli dgn Header Digital', df_asli_t['digital_annotation'] == 1),
    ('Asli Bersih', (df_asli_t['handwritten_annotation'] == 0) & (df_asli_t['digital_annotation'] == 0)),
]:
    sub = df_asli_t[cond]
    fp_c = int((sub['pred'] == 1).sum())
    fp_rows.append({'Kelompok': lbl, 'Total': len(sub), 'FP': fp_c, 'FPR (%)': round(fp_c / max(len(sub), 1) * 100, 2)})
print("=== Audit False Positive pada Struk Asli (Slide 15) ===")
display(pd.DataFrame(fp_rows))

---
## Bagian 6 — Galeri *Before & After* Hasil Deteksi (`TP`, `FP`, `FN`) & Export `config_final.json`

Untuk memudahkan presentasi dan penulisan laporan UAS, kita tampilkan berdampingan **BEFORE (Citra Input Polos)** vs **AFTER (Hasil Deteksi Sistem: Kotak Merah + Garis Hubung Copy-Paste + Kotak Hijau Putus-Putus Ground Truth)**.

In [ ]:
def plot_before_after_detection(row, tag: str):
    img_rgb = cv2.cvtColor(cv2.imread(row['filepath']), cv2.COLOR_BGR2RGB)
    p_obj = pred_obj_map[row['filename']]
    gt_regs = df_test_mod[df_test_mod['filename'] == row['filename']]

    fig, axes = plt.subplots(1, 2, figsize=(11, 6.2))
    axes[0].imshow(img_rgb)
    axes[0].set_title(f"BEFORE — Input Struk ({row['filename']})")
    axes[0].axis('off')

    axes[1].imshow(img_rgb)
    # Ground truth (hijau putus-putus)
    for _, gr in gt_regs.iterrows():
        axes[1].add_patch(patches.Rectangle((gr['x'], gr['y']), gr['width'], gr['height'],
                                            linewidth=2.2, edgecolor='#16a34a', facecolor='none', linestyle='--'))
    # Garis penghubung pasangan CPI (oranye) + kotak prediksi (merah)
    for cp in p_obj['cpi_pairs'][:8]:
        x1, y1, w1, h1 = cp['bbox1']
        x2, y2, w2, h2 = cp['bbox2']
        axes[1].plot([x1 + w1/2, x2 + w2/2], [y1 + h1/2, y2 + h2/2], color='#f59e0b', linewidth=1.4, alpha=0.8)
    for pb in p_obj['boxes'][:16]:
        bx, by, bw, bh = pb['bbox']
        axes[1].add_patch(patches.Rectangle((bx, by), bw, bh, linewidth=2.2, edgecolor='#dc2626', facecolor='none'))

    verdict = 'TERINDIKASI DIUBAH' if p_obj['pred'] == 1 else 'ASLI'
    axes[1].set_title(f"AFTER ({tag}) — {verdict}\nCPI={p_obj['n_cpi']} pasang | IMI={p_obj['n_imi']} | Merah=Prediksi, Hijau=GT")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

tp_df = df_test_eval[(df_test_eval['forged'] == 1) & (df_test_eval['pred'] == 1)]
fp_df = df_test_eval[(df_test_eval['forged'] == 0) & (df_test_eval['pred'] == 1)]
fn_df = df_test_eval[(df_test_eval['forged'] == 1) & (df_test_eval['pred'] == 0)]

print("=== Contoh TRUE POSITIVE (Berhasil Menangkap & Menandai Nominal/Tanggal Palsu) ===")
for _, r in tp_df.head(2).iterrows():
    plot_before_after_detection(r, 'TRUE POSITIVE')

if len(fp_df) > 0:
    print("=== Satu-Satunya FALSE POSITIVE di TEST (Pinggiran Kertas Kuning di Tepi Kanan) ===")
    plot_before_after_detection(fp_df.iloc[0], 'FALSE POSITIVE')

if len(fn_df) > 0:
    print("=== Contoh FALSE NEGATIVE (Kasus Lolos / Keterbatasan Sistem) ===")
    plot_before_after_detection(fn_df.iloc[0], 'FALSE NEGATIVE')

# Export konfigurasi final untuk aplikasi Streamlit
final_cfg_path = PROJECT_DRIVE_DIR / 'config_final.json'
with open(final_cfg_path, 'w', encoding='utf-8') as f:
    json.dump({'best_cfg': best_cfg, 'val_metrics': best_val_metrics, 'test_metrics': m_test}, f, indent=2)
print(f"Konfigurasi final disimpan ke: {final_cfg_path}")

---
## Ringkasan Akhir Proyek

### Q&A
- **Apakah metode OpenCV klasik tanpa YOLO/Deep Learning mampu mengungguli baseline paper ICDAR 2023 pada split `test` (218 struk, 35 palsu)?** Ya — pipeline klasik 8 tahap menghasilkan **F1 kelas palsu `77,97%`** (`Precision 95,83%`, `Recall 65,71%`), jauh melampaui SVM (`13,33%`), BoW+LR (`17,78%`), ChatGPT strict (`25,20%`), dan ChatGPT relaxed (`28,39%`).

### Data Analysis Key Findings
- **Audit Dataset:** 60,6% citra *Find it! Again* adalah Grayscale (hanya 39,4% RGB), sehingga fitur kromatik tidak cukup berdiri sendiri; 1 file (`X51006619709.png`) hilang di folder `val/` dan dikecualikan agar evaluasi bersih.
- **Target Utama (`Total/payment`):** Sistem mencapai **Recall `92,86%`** (13 dari 14 struk dengan nominal total diubah berhasil terdeteksi), dan **64,47%** kotak manipulasi `CPI` tertimpa langsung oleh kotak prediksi (`IoU ≥ 0,10`).
- **Ketahanan terhadap Gangguan Asli:** Pada 183 struk asli di `test`, hanya terjadi **1 False Positive (`0,55%` FPR)** yang disebabkan tepi kertas karbon kuning di luar badan struk; **0 FP** terjadi pada 11 struk ber-header digital.

### Insights or Next Steps
- **Keterbatasan yang jujur:** Deteksi berfokus pada *copy-paste* internal (`CPI`, Recall `77,78%`) dan ketik ulang (`IMI`, Recall `66,67%`); pemalsuan salin dari struk lain (`CPO`, `0%`) dan penghapusan murni (`CUT`, `40%`) masih menjadi ruang pengembangan.
- **Langkah Akhir:** Konfigurasi `config_final.json` siap dibungkus ke aplikasi web **Streamlit** agar pengguna dapat mengunggah citra struk dan melihat hasil deteksi serta visualisasi tiap tahap PCD secara interaktif.